# Privacy Archetypes

Analysis for multi-metric clustering and PCA visualization of privacy archetypes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

metrics_results = Path("../data/mhealth_apps_metrics.csv")
if not metrics_results.exists():
    raise FileNotFoundError(
        "Could not find `mhealth_apps_metrics.csv` in the notebook folder or at ../data/."
    )

REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", 
    "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", 
    "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", 
    "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", 
    "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", 
    "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", 
    "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina",
    "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain",
    "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia",
    "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore",
    "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria",
    "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand",
    "pg": "Papua New Guinea"
}

REGION_ORDER = [
    "North America", "Latin America", "Europe",
    "Middle East", "Asia", "Africa", "Oceania"
]

region_palette = plt.cm.tab10(np.linspace(0, 1, len(REGION_ORDER)))
REGION_COLORS = {region: region_palette[i] for i, region in enumerate(REGION_ORDER)}
category_palette = plt.cm.tab20(np.linspace(0, 1, 20))


In [ ]:
import pandas as pd

df = pd.read_csv(metrics_results, low_memory=False)
print("Raw shape:", df.shape)

required_metrics = ["ADII", "DGI", "PCLR", "AS"]
missing_required = [c for c in required_metrics if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required metric columns: {missing_required}")

analysis_df = df.copy()

print("Analysis shape:", analysis_df.shape)
print("Unique apps:", analysis_df["app_id"].nunique())

display_cols = [
    "app_id", "country", "country_label", "region", "category",
    "ADII", "DGI", "PCLR", "AS",
    "observed_count", "disclosed_count", "missing_count", "misleading_count"
]
display_cols = [c for c in display_cols if c in analysis_df.columns]

In [ ]:

summary = pd.DataFrame({
    "non_null": analysis_df[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": analysis_df[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": analysis_df[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": analysis_df[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": analysis_df[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": analysis_df[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)
summary


In [ ]:

country_level = analysis_df.copy()

metric_agg = {
    "ADII": "mean",
    "DGI": "mean",
    "PCLR": "mean",
    "AS": "mean",
}

meta_agg = {
    "country_label": "nunique",
    "region": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "category": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "num_permissions": "mean",
    "num_dangerous_permissions": "mean",
    "num_trackers": "mean",
    "downloads_int": "mean",
    "average_score": "mean",
}

agg_dict = {}
for k, v in {**metric_agg, **meta_agg}.items():
    if k in analysis_df.columns:
        agg_dict[k] = v

app_level = analysis_df.groupby("app_id", as_index=False).agg(agg_dict)
app_level = app_level.rename(columns={"country_label": "country_count"})
app_level_all = app_level.copy()

if "PCLR" not in app_level_all.columns:
    if "app_country_PCLR" in analysis_df.columns:
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)["app_country_PCLR"]
            .mean()
            .rename(columns={"app_country_PCLR": "PCLR"})
        )
        app_level_all = app_level_all.merge(pclr_fallback, on="app_id", how="left")
    elif {"app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"}.issubset(analysis_df.columns):
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)[
                ["app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"]
            ]
            .mean()
        )
        denom = pclr_fallback["app_country_total_sensitive_instances"].replace(0, np.nan)
        pclr_fallback["PCLR"] = pclr_fallback["app_country_pre_sensitive_instances"] / denom
        app_level_all = app_level_all.merge(pclr_fallback[["app_id", "PCLR"]], on="app_id", how="left")

print("Country-level rows:", len(country_level))
print("App-level rows:", len(app_level_all))
print("App-level metric columns:", [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in app_level_all.columns])
app_level = app_level_all.copy()




## Privacy archetypes from multi-metric clustering

This section clusters apps using `ADII`, `DGI`, `PCLR`, and `AS`, then labels each cluster by its dominant privacy pattern.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cluster_source = (
    app_level_all.copy()
    if "app_level_all" in globals()
    else app_level.copy()
    if "app_level" in globals()
    else analysis_df.copy()
)

metric_aliases = {
    "ADII": ["ADII", "ADII_mean", "ADII_median"],
    "DGI": ["DGI", "DGI_mean", "DGI_median"],
    "PCLR": ["PCLR", "PCLR_mean", "PCLR_median"],
    "AS": ["AS", "AS_mean", "AS_median"],
    "app_id": ["app_id"],
    "category": ["category"],
    "num_trackers": ["num_trackers"],
    "num_permissions": ["num_permissions"],
    "downloads_int": ["downloads_int", "downloads"],
}

cluster_df = pd.DataFrame(index=cluster_source.index)

for target, candidates in metric_aliases.items():
    for cand in candidates:
        if cand in cluster_source.columns:
            cluster_df[target] = cluster_source[cand]
            break

feature_cols = ["ADII", "DGI", "PCLR", "AS"]
needed = ["app_id"] + feature_cols

missing = [c for c in needed if c not in cluster_df.columns]
if missing:
    raise ValueError(
        f"Clustering requires {needed}. "
        f"Missing: {missing}. "
        f"Available columns: {cluster_source.columns.tolist()}"
    )


for c in feature_cols + ["num_trackers", "num_permissions", "downloads_int"]:
    if c in cluster_df.columns:
        cluster_df[c] = pd.to_numeric(cluster_df[c], errors="coerce")

n_before_dropna = len(cluster_df)
cluster_df = cluster_df.dropna(subset=feature_cols).copy()
n_excluded = n_before_dropna - len(cluster_df)
print(
    f"Apps with complete {feature_cols} metrics: {len(cluster_df)}/{n_before_dropna} "
    f"({n_excluded} excluded, typically apps observed in too few countries for AS to be defined)."
)

if cluster_df["app_id"].duplicated().any():
    agg = {
        "ADII": "mean",
        "DGI": "mean",
        "PCLR": "mean",
        "AS": "mean",
    }

    for extra in ["category", "num_trackers", "num_permissions", "downloads_int"]:
        if extra in cluster_df.columns:
            if pd.api.types.is_numeric_dtype(cluster_df[extra]):
                agg[extra] = "mean"
            else:
                agg[extra] = "first"

    cluster_df = cluster_df.groupby("app_id", as_index=False).agg(agg)

if len(cluster_df) < 3:
    raise ValueError("Need at least 3 apps with complete metrics for clustering.")

# ADII is strongly right-skewed (see 11_regression_analysis.ipynb, which
# log-transforms it for the same reason). Feeding raw ADII into
# StandardScaler only rescales by the (outlier-inflated) mean/std -- it does
# not compress the tail, so a handful of extreme-ADII apps end up with
# z-scores an order of magnitude larger than every other feature and can
# dominate the Euclidean distances K-means uses. Clustering on log1p(ADII)
# avoids that, while the cluster profile below still reports/labels using
# the original ADII scale for interpretability.
cluster_df["log_ADII"] = np.log1p(cluster_df["ADII"])
cluster_feature_cols = ["log_ADII", "DGI", "PCLR", "AS"]

X = cluster_df[cluster_feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# KMeans clustering
n_clusters = 4 if len(cluster_df) >= 40 else min(3, max(2, len(cluster_df) // 10 or 2))

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=20,
)

cluster_df["archetype_id"] = kmeans.fit_predict(X_scaled)

# Diagnostic: silhouette score for the chosen k, plus scores for a small
# range of alternative k values, since k=4 (or the sample-size-based
# fallback) was previously chosen by convention rather than validated.
sil_score = silhouette_score(X_scaled, cluster_df["archetype_id"])
print(f"\nSilhouette score for k={n_clusters}: {sil_score:.3f}")

max_k_to_check = min(8, len(cluster_df) - 1)
if max_k_to_check >= 2:
    print("Silhouette score by k (for reference):")
    for k in range(2, max_k_to_check + 1):
        if k == n_clusters:
            continue
        trial_labels = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X_scaled)
        print(f"  k={k}: {silhouette_score(X_scaled, trial_labels):.3f}")


# Cluster profiles (reported on the original ADII scale, not log_ADII, for
# interpretability -- only the clustering itself uses the log-transformed
# feature).
cluster_profile = (
    cluster_df.groupby("archetype_id")[feature_cols]
    .mean()
    .round(4)
)

cluster_profile_z = pd.DataFrame(
    scaler.transform(cluster_df.groupby("archetype_id")[cluster_feature_cols].mean()),
    index=cluster_profile.index,
    columns=[f"{c}_z" for c in cluster_feature_cols],
)

name_map = {}

adii_median = cluster_profile["ADII"].median()
dgi_median = cluster_profile["DGI"].median()
pclr_median = cluster_profile["PCLR"].median()
as_median = cluster_profile["AS"].median()

adii_q75 = cluster_profile["ADII"].quantile(0.75)
as_q75 = cluster_profile["AS"].quantile(0.75)

for cid, row in cluster_profile.iterrows():
    adii = row["ADII"]
    dgi = row["DGI"]
    pclr = row["PCLR"]
    ascore = row["AS"]

    if adii > adii_median and dgi > dgi_median:
        base = "Opaque high-collection"
    elif pclr > pclr_median:
        base = "Consent-weak collectors"
    elif ascore > as_median:
        base = "Geographically adaptive"
    else:
        base = "Lower-intensity / more stable"

    if base == "Opaque high-collection":
        variant = "extreme collection" if adii > adii_q75 else "elevated collection"

    elif base == "Consent-weak collectors":
        variant = "high intensity" if adii > adii_q75 else "moderate intensity"

    elif base == "Geographically adaptive":
        variant = "high adaptation" if ascore > as_q75 else "moderate adaptation"

    else:
        variant = "under-reporting bias" if dgi > dgi_median else "low exposure"

    name_map[cid] = f"{base} ({variant})"


cluster_df["privacy_archetype"] = cluster_df["archetype_id"].map(name_map)
cluster_profile["privacy_archetype"] = cluster_profile.index.map(name_map)

display(cluster_profile.reset_index())

print("\nArchetype counts:")
print(cluster_df["privacy_archetype"].value_counts())


In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)

plot_df = cluster_df.copy()
plot_df["PC1"] = coords[:, 0]
plot_df["PC2"] = coords[:, 1]

short_name_map = {
    "Consent-weak collectors (moderate intensity)": "Consent-weak",
    "Consent-weak collectors (high intensity)": "Consent-weak",

    "Geographically adaptive (moderate adaptation)": "Geo-adaptive",
    "Geo-adaptive": "Geo-adaptive",

    "Lower-intensity / more stable (under-reporting bias)": "Stable / lower-intensity",
    "Stable / lower-intensity": "Stable / lower-intensity",

    "Opaque high-collection (extreme collection)": "Opaque high-collection",
    "Opaque high-collection (elevated collection)": "Opaque high-collection",
}

plot_df["archetype_short"] = (
    plot_df["privacy_archetype"]
    .map(short_name_map)
    .fillna(plot_df["privacy_archetype"])
)

palette = {
    "Consent-weak": "#1F77B4",              # blue
    "Geo-adaptive": "#FF7F0E",              # orange
    "Stable / lower-intensity": "#2CA02C", # green
    "Opaque high-collection": "#D62728",   # red
}

def add_confidence_ellipse(ax, x, y, color, n_std=1.5, alpha=0.12):
    if len(x) < 3:
        return
    cov = np.cov(x, y)
    if np.any(~np.isfinite(cov)):
        return

    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    if np.any(vals <= 0):
        return

    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    width, height = 2 * n_std * np.sqrt(vals)

    ellipse = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=theta,
        facecolor=color,
        edgecolor=color,
        lw=1.2,
        alpha=alpha,
        zorder=1
    )
    ax.add_patch(ellipse)

# Custom legend entry for centroid (X marker)
centroid_handle = Line2D(
    [0], [0],
    marker='X',
    color='w',
    markerfacecolor='gray',
    markeredgecolor='black',
    markersize=10,
    linewidth=0,
    label='Cluster centroid (mean PC1, PC2)'
)

fig, ax = plt.subplots(figsize=(8, 3))

for label, subset in plot_df.groupby("archetype_short"):
    color = palette.get(label, "#777777")

    add_confidence_ellipse(ax, subset["PC1"], subset["PC2"], color=color, n_std=1.5, alpha=0.12)

    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        s=34,
        alpha=0.65,
        color=color,
        edgecolor="white",
        linewidth=0.4,
        label=f"{label} (n={len(subset)})",
        zorder=2
    )

    # centroid
    cx, cy = subset["PC1"].mean(), subset["PC2"].mean()
    ax.scatter(
        cx, cy,
        s=140,
        color=color,
        edgecolor="black",
        linewidth=1.0,
        marker="X",
        zorder=3
    )

ax.axhline(0, color="lightgray", lw=0.8, linestyle="--", zorder=0)
ax.axvline(0, color="lightgray", lw=0.8, linestyle="--", zorder=0)

LABEL_FONTSIZE = 12
TICK_FONTSIZE = 12

ax.set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)",
    fontsize=LABEL_FONTSIZE
)
ax.set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)",
    fontsize=LABEL_FONTSIZE
)

ax.tick_params(axis='both', which='major', labelsize=TICK_FONTSIZE)

ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

handles, labels = ax.get_legend_handles_labels()
handles.append(centroid_handle)
labels.append('Cluster centroid (mean PC1, PC2)')

ax.legend(
    handles,
    labels,
    frameon=False,
    loc="upper left",
    bbox_to_anchor=(1, 0.8),
    title="Archetypes",
    fontsize=12,
    title_fontsize=12
)

plt.tight_layout()
plt.savefig("../figures/privacy_archetypes_pca.png", dpi=300, bbox_inches="tight")
plt.show()